# 探索的データ分析（EDA）


このノートブックでは`data_recording.py`によって収集した全15セッション分のデータの品質を統計検定を用いて分析しています。


## 集中・非集中ラベル間のEEGパワー差の統計分析


今回はデータに含まれるノイズが検定結果に及ぼす影響を明らかにするため、全15セッション分のデータを含むフルデータセットに加え、
信号品質指標に基づきノイズが多いと判定したデータを除外したクリーンデータセットを追加作成し、検定結果を比較しました。

まず、データの正規性の判定をシャピロウィルク検定を用いて実施し、検定方法を決定しました。

25特徴量それぞれについて、対応のあるt検定を実施し、
さらに多重比較に対応するためBenjamini-Hochberg法によるFDR（偽発見率）補正を適用しました。

FDR補正後に有意となった特徴量は残念ながらありませんでした（補正後p値はすべて > 0.05）。
12セッション、単一被験者デザインのデータ不足による検出力の限界によるものだと考えられます。

効果量（cohen's d）が0.80を超え、かつ未補正のp値も0.05未満である、
有意な価値を示す可能性のある特徴量について、最終的な考察を加えました。

### 前処理済みデータの読み込み


In [1]:
import pandas as pd
from pathlib import Path

PROJECT_ROOT = Path().resolve().parent
DATA_PATH = PROJECT_ROOT / "data" / "processed_by_session" / "all_sessions_normalized.csv"
df = pd.read_csv(DATA_PATH)
df.head()

,TP9_theta,TP9_alpha,TP9_beta,TP9_theta_alpha,TP9_beta_alpha,TP9_engagement,AF7_theta,AF7_alpha,AF7_beta,AF7_theta_alpha,...,AF8_engagement,TP10_theta,TP10_alpha,TP10_beta,TP10_theta_alpha,TP10_beta_alpha,TP10_engagement,alpha_asymmetry,label,session
0,1.234471,1.210795,0.978582,0.982733,0.776915,0.851526,1.034322,0.765393,0.863495,1.222779,...,1.181369,1.001467,1.176479,1.089762,0.859498,0.898335,0.992781,-0.554765,1,1
1,0.678074,0.908069,1.327298,0.719753,1.405066,1.727267,0.988014,0.904989,0.983758,0.987863,...,1.013922,0.856766,0.952863,0.925597,0.907871,0.942069,1.016758,1.395709,1,1
2,1.099751,1.048086,1.468173,1.011400,1.346566,1.458644,1.030313,0.818970,1.020682,1.138356,...,0.987976,0.751295,0.777029,1.040647,0.976261,1.298846,1.356943,1.950642,1,1
3,1.033908,1.143660,1.233292,0.871385,1.036612,1.190827,0.743382,0.578643,0.873885,1.162460,...,0.700564,0.878124,0.996854,1.010756,0.889440,0.983345,1.070851,-3.202137,1,1
4,0.758928,1.115262,1.273642,0.655917,1.097786,1.390569,0.773703,0.711743,0.928365,0.983621,...,1.030596,0.823808,1.215817,0.876779,0.684149,0.699380,0.846399,-0.242200,1,1


## ノイズの多いデータを除外した「クリーン」データセットの作成


パーソナル脳波測定デバイスで測定されたデータにはノイズが多く含まれ、本プロジェクトにおいてノイズの除去とデータ品質の保護のバランスが大きな懸念点でした。
そのため全１５セッションを含むデータセットに加え、ノイズが多いデータを除いた「クリーン」なデータセットを作成し、それぞれの効果を比較することにしました。

### セッション除外基準

セッション品質分析の際に計算した、以下の事前定義済みしきい値のうち
2つ以上に該当したセッションを除外対象としてフラグ付けしました：

- 全特徴量の平均変動係数（CV） > 0.6
- Outlier率（いずれかの特徴量が3σを超えるサンプルの割合） > 15%
- ラベルバランス比（少数派/多数派クラス） < 0.6
- 平均theta/alpha比 > 1.10（眠気の可能性）


In [2]:
import pandas as pd
import numpy as np

df = pd.read_csv("../data/processed_by_session/all_sessions_normalized.csv")

feature_cols = [c for c in df.columns if c not in ['label', 'session']]
theta_alpha_cols = [c for c in feature_cols if 'theta_alpha' in c]

results = []

for session_id in sorted(df['session'].unique()):
    sub = df[df['session'] == session_id]
    
    # Mean CV
    cvs = []
    for col in feature_cols:
        mean_val = sub[col].mean()
        std_val = sub[col].std()
        cv = std_val / abs(mean_val) if mean_val != 0 else np.nan
        cvs.append(cv)
    mean_cv = np.nanmean(cvs)
    
    # Outlier Ratio (|zscore| > 3 sample proportion)
    z_scores = sub[feature_cols].apply(lambda col: np.abs((col - col.mean()) / col.std()))
    outlier_mask = (z_scores > 3).any(axis=1)
    outlier_rate = outlier_mask.mean()
    
    # Label imbalance (ratio: smaller / larger)
    label_counts = sub['label'].value_counts()
    label_ratio = label_counts.min() / label_counts.max()
    
    # Mean theta/alpha ratio
    theta_alpha_mean = sub[theta_alpha_cols].mean().mean()
    
    results.append({
        'session': session_id,
        'n_samples': len(sub),
        'mean_cv': round(mean_cv, 3),
        'outlier_rate': round(outlier_rate, 3),
        'label_ratio': round(label_ratio, 3),
        'theta_alpha_mean': round(theta_alpha_mean, 3),
    })

quality_df = pd.DataFrame(results)

# Flag sessions outside of threshold
quality_df['flag_cv']       = quality_df['mean_cv'] > 0.6
quality_df['flag_outlier']  = quality_df['outlier_rate'] > 0.15
quality_df['flag_imbalance'] = quality_df['label_ratio'] < 0.6
quality_df['flag_drowsy']   = quality_df['theta_alpha_mean'] > 1.1

quality_df['flag_count'] = quality_df[
    ['flag_cv', 'flag_outlier', 'flag_imbalance', 'flag_drowsy']
].sum(axis=1)

print(quality_df.to_string(index=False))

# Show sessions to remove
excluded_sessions = quality_df[quality_df['flag_count'] >= 2]['session'].tolist()
print(f"\nSessions to Remove: {excluded_sessions}")

 session  n_samples  mean_cv  outlier_rate  label_ratio  theta_alpha_mean  flag_cv  flag_outlier  flag_imbalance  flag_drowsy  flag_count
       1        282    0.342         0.181        0.880             1.038    False          True           False        False           1
       2        205    0.502         0.132        0.627             0.998    False         False           False        False           0
       3        242    0.524         0.153        0.984             1.069    False          True           False        False           1
       4        255    0.890         0.157        0.848             0.976     True          True           False        False           2
       5        297    0.402         0.125        0.980             1.040    False         False           False        False           0
       6        251    0.370         0.112        0.992             1.075    False         False           False        False           0
       7        163    0.374      

| セッション | CV | Outlier率 | ラベル比率 | θ/α | フラグ |
|---------|-----|-----------|--------------|-----|-------|
| 4       | 0.89| 15.7%     | 85%          | 0.98| CV, outlier |
| 7       | 0.37| 12.9%     | 19%          | 1.12| label imbalance |
| 8       | 0.97| 11.6%     | 100%         | 1.14| CV, drowsy |

これに基づき、セッション4・7・8を「クリーン」な学習用データセットから除外しました。


In [3]:
clean_df = df[~(df['session'] == 4) & ~(df['session'] == 7) & ~(df['session'] == 8)]
print(clean_df['session'].unique())
print(len(clean_df))

[ 1  2  3  5  6  9 10 11 12 13 14 15]
3112


### セッション・条件別に分割


In [4]:
# full dataset ver
session_means = df.groupby(["session", "label"]).mean().reset_index()
focus = session_means[session_means['label'] == 1].reset_index(drop=True)
dist = session_means[session_means['label'] == 0].reset_index(drop=True)


In [5]:
# clean dataset ver
session_means = clean_df.groupby(["session", "label"]).mean().reset_index()
focus_clean = session_means[session_means['label'] == 1].reset_index(drop=True)
dist_clean = session_means[session_means['label'] == 0].reset_index(drop=True)

## 正規性の検定：シャピロウィルク検定

脳波データを用いた過去文献では脳波データは非正規性を示すことが多いと報告されていますが、本データセットにおける脳波データの正規性を調べ検定方法を決定するため、シャピロウィルク検定を実施しました。

### フルデータ

In [6]:
from scipy.stats import shapiro

normality_results = []

for feature in feature_cols:
    diff = focus[feature].values - dist[feature].values
    stat, p = shapiro(diff)
    normality_results.append({
        'feature': feature,
        'shapiro_stat': round(stat, 3),
        'shapiro_p': round(p, 4),
        'normal': p > 0.05
    })

normality_df = pd.DataFrame(normality_results)
n_non_normal = (~normality_df['normal']).sum()
print(f"正規性が棄却された特徴量: {n_non_normal} / {len(normality_df)}")
normality_df.sort_values('shapiro_p')

正規性が棄却された特徴量: 4 / 25


,feature,shapiro_stat,shapiro_p,normal
24,alpha_asymmetry,0.777,0.0019,False
7,AF7_alpha,0.859,0.0235,False
10,AF7_beta_alpha,0.875,0.0401,False
9,AF7_theta_alpha,0.877,0.0422,False
2,TP9_beta,0.885,0.0573,True
18,TP10_theta,0.902,0.1014,True
12,AF8_theta,0.910,0.1369,True
16,AF8_beta_alpha,0.919,0.1849,True
20,TP10_beta,0.924,0.2239,True
15,AF8_theta_alpha,0.925,0.2267,True


### クリーンデータ

In [7]:
from scipy.stats import shapiro

normality_results = []

for feature in feature_cols:
    diff = focus_clean[feature].values - dist_clean[feature].values
    stat, p = shapiro(diff)
    normality_results.append({
        'feature': feature,
        'shapiro_stat': round(stat, 3),
        'shapiro_p': round(p, 4),
        'normal': p > 0.05
    })

normality_df = pd.DataFrame(normality_results)
n_non_normal = (~normality_df['normal']).sum()
print(f"正規性が棄却された特徴量: {n_non_normal} / {len(normality_df)}")
normality_df.sort_values('shapiro_p')

正規性が棄却された特徴量: 2 / 25


,feature,shapiro_stat,shapiro_p,normal
24,alpha_asymmetry,0.800,0.0093,False
16,AF8_beta_alpha,0.836,0.0248,False
2,TP9_beta,0.864,0.0542,True
12,AF8_theta,0.875,0.0767,True
0,TP9_theta,0.910,0.2136,True
17,AF8_engagement,0.916,0.2528,True
18,TP10_theta,0.919,0.2762,True
15,AF8_theta_alpha,0.924,0.3219,True
10,AF7_beta_alpha,0.926,0.3380,True
22,TP10_beta_alpha,0.927,0.3516,True


結果、フルデータの場合25特徴量のうち21特徴量、クリーンデータでは23特徴量で、正規性からの有意な逸脱は検出されませんでした（p > 0.05）。
従って、本データセットでは対応のあるt検定を採用しました。ただし、n=12という少ないサンプルサイズでは正規性検定自体の検出力も限られるため、この判断には不確実性が残っています。

## 対応のあるｔ検定

25特徴量それぞれについて対応のあるt検定を実施し、p値、効果量 (cohen's d)を計算しました。
加えて、多重検定による偽陽性を抑えるためBenjamini-Hochberg 法によるFalse Discovery Rate(FDR)補正を適用したp-correctedを算出しました。

### フルデータ

In [8]:
import numpy as np
from statsmodels.stats.multitest import multipletests
from scipy.stats import ttest_rel

results = []
feature_cols = [col for col in df.columns if col not in ['session', 'label']]
for col in feature_cols:
    f_vals = focus[col].values
    d_vals = dist[col].values
    stat, p = ttest_rel(f_vals, d_vals) 

    n = len(f_vals)
    diff = f_vals - d_vals
    effect_size = np.mean(diff) / np.std(diff, ddof=1) 

    med_focus = np.median(f_vals)
    med_dist = np.median(d_vals)
    direction = "focus > distracted" if med_focus > med_dist else "distracted > focus"

    results.append({
        "feature":     col,
        "p_value":     p,
        "effect_size": round(effect_size, 3),
        "direction":   direction,
        "median_focus": round(med_focus, 4),
        "median_distracted": round(med_dist, 4)
    })

results_df = pd.DataFrame(results)

reject, p_corrected, _, _ = multipletests(results_df['p_value'], method='fdr_bh')
results_df['p_corrected'] = p_corrected.round(4)
results_df['significant'] = reject

print(results_df.sort_values('p_corrected').to_string(index=False))

         feature  p_value  effect_size          direction  median_focus  median_distracted  p_corrected  significant
        AF7_beta 0.011147       -0.754 distracted > focus        0.9644             0.9882       0.1393        False
 TP10_engagement 0.005983       -0.835 distracted > focus        1.0306             1.0864       0.1393        False
       TP10_beta 0.040325       -0.583 distracted > focus        0.9847             1.0286       0.3360        False
 TP10_beta_alpha 0.062530       -0.522 distracted > focus        1.0530             1.0797       0.3726        False
      TP10_theta 0.074525        0.498 focus > distracted        1.0029             0.9580       0.3726        False
       AF8_theta 0.122723        0.424 focus > distracted        1.0188             0.9714       0.3901        False
        TP9_beta 0.124834        0.422 focus > distracted        1.0952             1.0581       0.3901        False
TP10_theta_alpha 0.110709        0.440 focus > distracted       

### クリーンデータ

In [9]:
import numpy as np
from statsmodels.stats.multitest import multipletests
from scipy.stats import ttest_rel

results = []
feature_cols = [col for col in df.columns if col not in ['session', 'label']]
for col in feature_cols:
    f_vals = focus_clean[col].values
    d_vals = dist_clean[col].values
    stat, p = ttest_rel(f_vals, d_vals) 

    n = len(f_vals)
    diff = f_vals - d_vals
    effect_size = np.mean(diff) / np.std(diff, ddof=1) 

    med_focus = np.median(f_vals)
    med_dist = np.median(d_vals)
    direction = "focus > distracted" if med_focus > med_dist else "distracted > focus"

    results.append({
        "feature":     col,
        "p_value":     p,
        "effect_size": round(effect_size, 3),
        "direction":   direction,
        "median_focus": round(med_focus, 4),
        "median_distracted": round(med_dist, 4)
    })

results_df_clean = pd.DataFrame(results)

reject, p_corrected, _, _ = multipletests(results_df_clean['p_value'], method='fdr_bh')
results_df_clean['p_corrected'] = p_corrected.round(4)
results_df_clean['significant'] = reject
print(results_df.sort_values('p_corrected').to_string(index=False))

         feature  p_value  effect_size          direction  median_focus  median_distracted  p_corrected  significant
        AF7_beta 0.011147       -0.754 distracted > focus        0.9644             0.9882       0.1393        False
 TP10_engagement 0.005983       -0.835 distracted > focus        1.0306             1.0864       0.1393        False
       TP10_beta 0.040325       -0.583 distracted > focus        0.9847             1.0286       0.3360        False
 TP10_beta_alpha 0.062530       -0.522 distracted > focus        1.0530             1.0797       0.3726        False
      TP10_theta 0.074525        0.498 focus > distracted        1.0029             0.9580       0.3726        False
       AF8_theta 0.122723        0.424 focus > distracted        1.0188             0.9714       0.3901        False
        TP9_beta 0.124834        0.422 focus > distracted        1.0952             1.0581       0.3901        False
TP10_theta_alpha 0.110709        0.440 focus > distracted       

## 結果


FDR補正後のp値は全ての特徴量において0.05を大幅に上回り、有意性を検出することはできませんでした。
従って今回取得した脳波データセットにおいて集中・非集中のラベル間で統計的に有意な差があるとは言えないという結論となりました。

### 留意点

In [10]:
#フルデータ
results_df[(results_df['p_value'] < 0.05) & (abs(results_df['effect_size']) > 0.80)].sort_values('p_value')

,feature,p_value,effect_size,direction,median_focus,median_distracted,p_corrected,significant
23,TP10_engagement,0.005983,-0.835,distracted > focus,1.0306,1.0864,0.1393,False


In [11]:
#クリーンデータ
results_df_clean[(results_df_clean['p_value'] < 0.05) & (abs(results_df_clean['effect_size']) > 0.80)].sort_values('p_value')

,feature,p_value,effect_size,direction,median_focus,median_distracted,p_corrected,significant
8,AF7_beta,0.017345,-0.808,distracted > focus,0.9854,0.9995,0.2624,False


フルデータセットを基にしたt検定ではTP10 engagementのみが、クリーンデータセットを使用したt検定ではAF7 betaのみがp < 0.05 かつ |d| > 0.8の有意な差の可能性を示唆する特徴量として抽出されました。

しかし、どちらの特徴量についても、集中状態ではβ波の動きが優位となるといった過去の知見とは逆方向の結果を示しています。

AF7 betaについてはAF7電極は側頭筋の近くに位置しており、特にタスク中の歯の食いしばりが強まる場面では筋電図（EMG）アーティファクトの影響を受けやすいため、タスク中の緊張状態が反映された可能性や、
TP10 engagementに関しては個人特有の脳波パターンを反映している可能性が考えられますが、この結果に関してはデータ数を増やした上で再検証する必要があります。

また信号品質が悪いデータを除去するかしないかで、有意な特徴量の数に違いはなく、有意性の可能性が示唆された特徴量が変わるという結果になりました。
信号品質が悪いデータの中にも集中・非集中ラベル間の効果の強さに影響が大きいデータが含まれていたことが原因と考えられ、ノイズ除去を慎重に行う必要性を強調する結果となりました。

また、信号品質が悪いデータを除去した場合、3セッションの除外により、すでに小さいサンプルサイズ（n=15）がさらに20%減少しました（n=12）。
検出力がそもそも限られている状況下で、ノイズの低減とサンプルサイズの確保がトレードオフの関係にあることを踏まえ、両方のデータセットを併用して結果を比較する今回のアプローチは妥当だったと考えられます。
